# XGBoost - Column Subsampling Trial

Controlled one-factor-at-a-time experiment for **colsample_bytree**. All other model settings are fixed to the stated baseline values.

**Important:** This notebook uses the validation split only for this experiment. The held-out test split is not evaluated here.

## 2. Package Installation


This cell checks whether the libraries required by the XGBoost workflow are installed and installs only the missing packages before the remaining audio-processing, analysis, plotting, and modelling steps run.

In [ ]:
# Purpose: Checks whether the libraries required by the cache-based XGBoost workflow are installed.
import importlib.util
import subprocess
import sys

required_packages = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("joblib", "joblib"),
    ("xgboost", "xgboost"),
]

missing = [pip_name for import_name, pip_name in required_packages if importlib.util.find_spec(import_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Required packages are already installed.")


## 3. Imports


This cell imports the standard-library and third-party tools used by the XGBoost workflow and provides a print-based fallback when the richer notebook display function is unavailable.

In [ ]:
# Purpose: Imports the standard-library and third-party tools used by the cache-based XGBoost workflow.
import json
import os
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

try:
    from IPython.display import display
except Exception:
    display = print


## 4. Configuration and Random Seeds


This cell fixes the available random seeds for reproducibility and defines the shared audio, MFCC, class, sampling, and XGBoost training settings used later in the notebook.

In [ ]:
# Purpose: Fixes the random seed and label mapping used by the XGBoost trial.
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}


## 5. Project and Output Paths


This cell defines and calls a portable helper that mounts Google Drive when Colab is available and otherwise continues without failing in a local environment.

In [4]:
# Purpose: Mounts Google Drive when this notebook is running in Google Colab.
def mount_drive_if_colab(mount_point="/content/drive"):
    try:
        from google.colab import drive
        drive.mount(mount_point)
        os.environ.setdefault(
            "INTRO_AI_PROJECT_ROOT",
            "/content/drive/MyDrive/Colab Notebooks/Education/INM701",
        )
        print("Google Colab detected. Google Drive mounted.")
    except Exception as exc:
        print("Google Colab Drive mount skipped. This is expected outside Colab.")
        print("Mount skip reason:", exc)

mount_drive_if_colab()
print("INTRO_AI_PROJECT_ROOT:", os.environ.get("INTRO_AI_PROJECT_ROOT", "not set"))


Mounted at /content/drive
Google Colab detected. Google Drive mounted.
INTRO_AI_PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Education/INM701


This cell resolves the project folder, locates the existing shared SVM feature cache, and creates the XGBoost output folders. It does not scan raw audio.

In [5]:
# Purpose: Resolve paths for cache-based XGBoost trials.
def resolve_project_root():
    candidates = []
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        candidates.append(Path(explicit).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        Path(r"C:/Users/PC/Documents/GitHub/Intro-to-AI"),
        Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701"),
    ])

    for candidate in candidates:
        if (candidate / "outputs" / "outputs" / "svm" / "cache").exists():
            return candidate
        if (candidate / "outputs" / "svm" / "cache").exists():
            return candidate
    return candidates[0]


def first_existing_path(candidates):
    expanded = [Path(candidate).expanduser() for candidate in candidates]
    for candidate in expanded:
        if candidate.exists():
            return candidate
    return expanded[0]


PROJECT_ROOT = resolve_project_root()
OUTPUTS_ROOT = first_existing_path([
    PROJECT_ROOT / "outputs" / "outputs",
    PROJECT_ROOT / "outputs",
])

SVM_CACHE_DIR = first_existing_path([
    OUTPUTS_ROOT / "svm" / "cache",
    PROJECT_ROOT / "outputs" / "svm" / "cache",
])

OUTPUT_DIR = OUTPUTS_ROOT / "xgboost"
FIGURES_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR = OUTPUT_DIR / "tables"

for directory in [OUTPUT_DIR, FIGURES_DIR, METRICS_DIR, MODELS_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Shared SVM cache:", SVM_CACHE_DIR)
print("XGBoost output directory:", OUTPUT_DIR)


Project root: /content/drive/MyDrive/Colab Notebooks/Education/INM701
Shared SVM cache: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/svm/cache
XGBoost output directory: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/xgboost


## 6. Load Shared SVM Cache


This cell loads the already prepared 80-dimensional MFCC mean/std feature cache from the SVM pipeline. It provides the same variable names used by the XGBoost trial cell, without rescanning audio, recomputing MFCCs, or refitting a scaler.

In [6]:
# Purpose: Load shared SVM MFCC cache instead of rescanning audio or extracting MFCCs again.
def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required cache file missing: {path}")
    return path

X_train_scaled = np.load(require_file(SVM_CACHE_DIR / "X_train.npy"), mmap_mode="r")
y_train = np.load(require_file(SVM_CACHE_DIR / "y_train.npy"))
X_validation_scaled = np.load(require_file(SVM_CACHE_DIR / "X_validation.npy"), mmap_mode="r")
y_validation = np.load(require_file(SVM_CACHE_DIR / "y_validation.npy"))

train_meta = pd.read_csv(require_file(SVM_CACHE_DIR / "train_metadata.csv"))
validation_meta = pd.read_csv(require_file(SVM_CACHE_DIR / "validation_metadata.csv"))

with open(require_file(SVM_CACHE_DIR / "feature_config.json"), "r", encoding="utf-8") as f:
    feature_config = json.load(f)

scaler = joblib.load(require_file(SVM_CACHE_DIR / "svm_standard_scaler.joblib"))

if X_train_scaled.shape[1] != 80 or X_validation_scaled.shape[1] != 80:
    raise RuntimeError("Expected 80-dimensional MFCC mean/std cache for XGBoost.")
if len(X_train_scaled) != len(y_train) or len(X_validation_scaled) != len(y_validation):
    raise RuntimeError("Feature and label arrays are not aligned.")

split_summary = pd.DataFrame({
    "split": ["train", "validation"],
    "bona_fide": [int((y_train == 0).sum()), int((y_validation == 0).sum())],
    "synthetic": [int((y_train == 1).sum()), int((y_validation == 1).sum())],
    "total": [len(y_train), len(y_validation)],
})
display(split_summary)

print("Loaded shared SVM feature cache:", SVM_CACHE_DIR)
print("Train:", X_train_scaled.shape, y_train.shape)
print("Validation:", X_validation_scaled.shape, y_validation.shape)
print("Feature representation:", feature_config.get("representation", "not recorded"))
print("Raw audio rescanned: No")
print("MFCCs recomputed here: No")
print("Scaler refitted here: No")


,split,bona_fide,synthetic,total
0,train,2798,6262,9060
1,validation,599,1343,1942


Loaded shared SVM feature cache: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/svm/cache
Train: (9060, 80) (9060,)
Validation: (1942, 80) (1942,)
Feature representation: aggregated MFCC mean+std (80 features), StandardScaler fitted on training only
Raw audio rescanned: No
MFCCs recomputed here: No
Scaler refitted here: No


## 7. Individual Hyperparameter Trial - `colsample_bytree`


In [7]:
# Purpose: Runs one controlled XGBoost hyperparameter trial while holding all other settings fixed.
from xgboost import XGBClassifier

# Only this hyperparameter changes in this notebook.
TRIAL_PARAMETER = 'colsample_bytree'
TRIAL_VALUES = [0.6, 0.8, 0.9, 1.0]

# Fixed baseline settings used for every candidate in this trial.
FIXED_PARAMS = dict(objective="binary:logistic", eval_metric="logloss", n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE, n_jobs=-1)

validation_results = []
models = {}

for value in TRIAL_VALUES:
    params = dict(FIXED_PARAMS)
    params[TRIAL_PARAMETER] = value
    model = XGBClassifier(**params)
    model.fit(X_train_scaled, y_train)

    validation_probability = model.predict_proba(X_validation_scaled)[:, 1]
    validation_pred = (validation_probability >= 0.5).astype(int)

    variant = f"{TRIAL_PARAMETER}={value}"
    row = {
        "model": 'XGBoost',
        "trial_parameter": TRIAL_PARAMETER,
        "trial_value": str(value),
        "variant": variant,
        "accuracy": accuracy_score(y_validation, validation_pred),
        "precision": precision_score(y_validation, validation_pred, zero_division=0),
        "recall": recall_score(y_validation, validation_pred, zero_division=0),
        "f1": f1_score(y_validation, validation_pred, zero_division=0),
    }
    row.update({"n_estimators": params["n_estimators"], "max_depth": params["max_depth"], "learning_rate": params["learning_rate"], "subsample": params["subsample"], "colsample_bytree": params["colsample_bytree"]})
    validation_results.append(row)
    models[variant] = model

validation_results_df = pd.DataFrame(validation_results).sort_values("f1", ascending=False).reset_index(drop=True)
display(validation_results_df)

best_variant = validation_results_df.iloc[0]["variant"]
best_model = models[best_variant]
print("Best validation result in this isolated trial:", best_variant)

trial_results_path = TABLES_DIR / "xgboost_colsample_bytree_trial.csv"
validation_results_df.to_csv(trial_results_path, index=False)
print("Saved trial results:", trial_results_path)


,model,trial_parameter,trial_value,variant,accuracy,precision,recall,f1,n_estimators,max_depth,learning_rate,subsample,colsample_bytree
0,XGBoost,colsample_bytree,1.0,colsample_bytree=1.0,0.934604,0.962006,0.942666,0.952238,200,4,0.05,0.9,1.0
1,XGBoost,colsample_bytree,0.8,colsample_bytree=0.8,0.929969,0.957544,0.940432,0.948911,200,4,0.05,0.9,0.8
2,XGBoost,colsample_bytree,0.9,colsample_bytree=0.9,0.929454,0.961715,0.935220,0.948282,200,4,0.05,0.9,0.9
3,XGBoost,colsample_bytree,0.6,colsample_bytree=0.6,0.925850,0.954511,0.937453,0.945905,200,4,0.05,0.9,0.6


Best validation result in this isolated trial: colsample_bytree=1.0
Saved trial results: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/xgboost/tables/xgboost_colsample_bytree_trial.csv
